In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install pathlib

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from pathlib import Path

In [ ]:
from google.colab import files
upload = files.upload()

Saving games.csv to games.csv


In [ ]:
#melakukan preprocessing data
games = pd.read_csv('games.csv')
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20058 entries, 0 to 20057
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              20058 non-null  object 
 1   rated           20058 non-null  bool   
 2   created_at      20058 non-null  float64
 3   last_move_at    20058 non-null  float64
 4   turns           20058 non-null  int64  
 5   victory_status  20058 non-null  object 
 6   winner          20058 non-null  object 
 7   increment_code  20058 non-null  object 
 8   white_id        20058 non-null  object 
 9   white_rating    20058 non-null  int64  
 10  black_id        20058 non-null  object 
 11  black_rating    20058 non-null  int64  
 12  moves           20058 non-null  object 
 13  opening_eco     20058 non-null  object 
 14  opening_name    20058 non-null  object 
 15  opening_ply     20058 non-null  int64  
dtypes: bool(1), float64(2), int64(4), object(9)
memory usage: 2.3+ MB


untuk mengecek apakah ada missing values.

In [ ]:
pd.DataFrame({'dtypes': games.dtypes, 'null_count': games.isnull().sum()}).loc[lambda x: x.null_count != 0]

,dtypes,null_count


In [ ]:
#pada dataset, tidak terdapat missing values
#maka selanjutnya dapat dilakukan transformasi variabel yang mengandung non-numerikal menjadi numerikal.

# Rated = 1, Not rated = 0
games['rated_bin'] = np.where(games['rated'] == True, 1, 0)
## White = 1, Black = 0
games['winner_bin'] = np.where(games['winner'] == 'white', 1, 0)

In [ ]:
#untuk mempermudah pembacaan saat exploratory data, kolom created_at dan last_move_at ke format yang mudah dibaca

games['created_at_dt'] = pd.to_datetime(games['created_at']/1000,
                                        unit='s',
                                        origin='unix')

games['last_move_at_dt'] = pd.to_datetime(games['last_move_at']/1000,
                                          unit='s',
                                          origin='unix')

In [ ]:
games['game_len_dt'] = games['last_move_at_dt'] - games['created_at_dt'
games['game_len'] = games['last_move_at'] - games['created_at']

games['game_len_mins'] = games['game_len'].apply(lambda x: round(x / 1000 / 60, 2))

In [ ]:
#jarak rating pemain akan mengukur perbedaan absolut rating di antara pemain di permainan tertentu

games['rating_distance'] = games['white_rating'] - games['black_rating']
games['rating_distance'] = games['rating_distance'].apply(abs)

In [ ]:
print(games['victory_status'].unique())
print(pd.factorize(games['victory_status'].unique())[0])

['outoftime' 'resign' 'mate' 'draw']
[0 1 2 3]


In [ ]:
#menggunakan One Hot Encoder
ohe = OneHotEncoder(handle_unknown='error')

In [ ]:
#menetapkan kolom transformator
trans_kol = make_column_transformer((ohe,['victory_status']),
                                       remainder='passthrough')

In [ ]:
trans_kol.fit(pd.DataFrame(games['victory_status']))

ColumnTransformer(remainder='passthrough',
                  transformers=[('onehotencoder', OneHotEncoder(),
                                 ['victory_status'])])

In [ ]:
## lakukan One-hot-encode pada variabel 'victory_status'  dan masukkan ke np.array()
hot_victory = trans_kol.transform(pd.DataFrame(games['victory_status'])).toarray()

In [ ]:
#membuat dataframe dari output transformator
VICTORY_STATUS = pd.DataFrame(hot_victory,
                             columns = trans_kol.get_feature_names_out())

In [ ]:
# me-rename kolom agar lebih jelas
VICTORY_STATUS.columns = ['victory_status_draw', 'victory_status_mate', 'victory_status_outoftime', 'victory_status_resign']

In [ ]:
# Merge hasil one-hot-encoding ke satu dataframe parent
games = games.merge(VICTORY_STATUS,
                    left_index=True,
                    right_index=True)

In [ ]:
#Sample hasil akhir dari one-hot-encoding
to_show = ['victory_status',
    'victory_status_draw',
    'victory_status_mate',
    'victory_status_outoftime',
    'victory_status_resign']

games[to_show].head(10)

,victory_status,victory_status_draw,victory_status_mate,victory_status_outoftime,victory_status_resign
0,outoftime,0.0,0.0,1.0,0.0
1,resign,0.0,0.0,0.0,1.0
2,mate,0.0,1.0,0.0,0.0
3,mate,0.0,1.0,0.0,0.0
4,mate,0.0,1.0,0.0,0.0
5,draw,1.0,0.0,0.0,0.0
6,resign,0.0,0.0,0.0,1.0
7,resign,0.0,0.0,0.0,1.0
8,resign,0.0,0.0,0.0,1.0
9,mate,0.0,1.0,0.0,0.0
